# 8-Puzzle Solver using A* Search Algorithm

This notebook implements the A* search algorithm to find the optimal sequence of moves to solve an 8-puzzle game. We use the **Manhattan Distance** heuristic.

In [5]:
import heapq
import copy

# Define the target goal configuration
GOAL_STATE = [
    [1, 2, 3],
    [8, 0, 4],
    [7, 6, 5]
]

class PuzzleNode:
    def __init__(self, board, parent=None, move="", g=0, h=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.g = g  # Path cost from start to current node
        self.h = h  # Heuristic cost (misplaced tiles)
        self.f = g + h  # Total evaluated cost

    # Overload the less-than operator so heapq can prioritize the lowest f(n)
    def __lt__(self, other):
        return self.f < other.f

def get_blank_position(board):
    """Finds the coordinates (row, col) of the blank tile (0)."""
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c
    return None

def calculate_misplaced_tiles(board, goal):
    """Calculates h(n): counts tiles that are not in their goal position."""
    count = 0
    for r in range(3):
        for c in range(3):
            # Misplaced metric usually ignores the blank space (0)
            if board[r][c] != 0 and board[r][c] != goal[r][c]:
                count += 1
    return count

def generate_neighbors(node):
    """Generates all valid adjacent board configurations by moving the blank tile."""
    neighbors = []
    r, c = get_blank_position(node.board)

    # Define valid directional moves for the blank space
    moves = {
        "Up": (-1, 0),
        "Down": (1, 0),
        "Left": (0, -1),
        "Right": (0, 1)
    }

    for direction, (dr, dc) in moves.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            # Create a deep copy of the board to modify
            new_board = copy.deepcopy(node.board)
            # Swap the blank tile with the target adjacent tile
            new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]

            neighbors.append(PuzzleNode(
                board=new_board,
                parent=node,
                move=direction,
                g=node.g + 1
            ))

    return neighbors

def board_to_tuple(board):
    """Converts the matrix state into an immutable tuple format for hashing."""
    return tuple(tuple(row) for row in board)

def print_board(board):
    """Utility helper to cleanly output the matrix rows."""
    for row in board:
        print("  ".join(str(x) if x != 0 else "_" for x in row))
    print()

def solve_a_star(initial_board):
    """Executes the A* Search algorithm to solve the puzzle."""
    start_node = PuzzleNode(
        board=initial_board,
        g=0,
        h=calculate_misplaced_tiles(initial_board, GOAL_STATE)
    )

    # Priority Queue (Min-Heap) for storing open states
    open_list = []
    heapq.heappush(open_list, start_node)

    # Set to keep track of visited board patterns
    closed_set = set()

    while open_list:
        current_node = heapq.heappop(open_list)

        # Check if the current node matches the goal state
        if current_node.board == GOAL_STATE:
            path = []
            while current_node:
                path.append(current_node)
                current_node = current_node.parent
            return path[::-1] # Reverse path to display start to goal

        board_tuple = board_to_tuple(current_node.board)
        if board_tuple in closed_set:
            continue
        closed_set.add(board_tuple)

        for neighbor in generate_neighbors(current_node):
            if board_to_tuple(neighbor.board) not in closed_set:
                neighbor.h = calculate_misplaced_tiles(neighbor.board, GOAL_STATE)
                neighbor.f = neighbor.g + neighbor.h
                heapq.heappush(open_list, neighbor)

    return None

# --- Execution Example ---
if __name__ == "__main__":
    # 0 represents the empty sliding tile
    initial_puzzle = [
        [2,8,3],
        [1,6,4],
        [7,0,5]
    ]

    print("--- Initial State ---")
    print_board(initial_puzzle)

    solution_path = solve_a_star(initial_puzzle)

    if solution_path:
        print(f"Goal state reached successfully in {len(solution_path) - 1} steps!\n")
        for step, node in enumerate(solution_path):
            if step > 0:
                print(f"Step {step}: Move Blank {node.move} (f={node.f}, g={node.g}, h={node.h})")
            else:
                print("Starting Position:")
            print_board(node.board)
    else:
        print("No solution exists for this configuration.")


--- Initial State ---
2  8  3
1  6  4
7  _  5

Goal state reached successfully in 5 steps!

Starting Position:
2  8  3
1  6  4
7  _  5

Step 1: Move Blank Up (f=4, g=1, h=3)
2  8  3
1  _  4
7  6  5

Step 2: Move Blank Up (f=5, g=2, h=3)
2  _  3
1  8  4
7  6  5

Step 3: Move Blank Left (f=5, g=3, h=2)
_  2  3
1  8  4
7  6  5

Step 4: Move Blank Down (f=5, g=4, h=1)
1  2  3
_  8  4
7  6  5

Step 5: Move Blank Right (f=5, g=5, h=0)
1  2  3
8  _  4
7  6  5



In [6]:
import copy
import random

def calculate_cost(state):
    """
    Calculates the heuristic cost h(n): the number of attacking pairs of queens.
    State representation: A list of 4 integers where index=column, value=row.
    """
    attacks = 0
    n = len(state)
    for i in range(n):
        for j in range(i + 1, n):
            # Check horizontal attacks (same row)
            if state[i] == state[j]:
                attacks += 1
            # Check diagonal attacks (|row_diff| == |col_diff|)
            elif abs(state[i] - state[j]) == abs(i - j):
                attacks += 1
    return attacks

def get_best_neighbors(state):
    """
    Generates all 12 possible neighbors by moving one queen per column
    and returns a list of neighbors that share the absolute lowest cost.
    """
    current_cost = calculate_cost(state)
    lowest_cost = float('inf')
    best_neighbors = []
    n = len(state)

    for col in range(n):
        original_row = state[col]
        for row in range(n):
            if row == original_row:
                continue  # Skip current position

            # Create neighbor state
            neighbor = list(state)
            neighbor[col] = row
            neighbor_cost = calculate_cost(neighbor)

            # Track neighbor with lowest cost
            if neighbor_cost < lowest_cost:
                lowest_cost = neighbor_cost
                best_neighbors = [neighbor]
            elif neighbor_cost == lowest_cost:
                best_neighbors.append(neighbor)

    return best_neighbors, lowest_cost

def print_board(state):
    """Prints a visual text representation of the 4x4 board."""
    for row in range(4):
        line = ""
        for col in range(4):
            if state[col] == row:
                line += " Q "
            else:
                line += " . "
        print(line)
    print()

def hill_climbing_4_queens(initial_state, max_sideways_moves=20):
    """
    Executes hill climbing search with a limit on continuous sideways moves
    to escape local shoulder plateaus.
    """
    current_state = list(initial_state)
    current_cost = calculate_cost(current_state)

    print("--- Start Initial State ---")
    print(f"State: {tuple(current_state)} | Cost (h): {current_cost}")
    print_board(current_state)

    sideways_count = 0
    step = 0

    while current_cost > 0:
        neighbors, lowest_neighbor_cost = get_best_neighbors(current_state)

        # Strict Hill Climbing Termination condition
        if lowest_neighbor_cost > current_cost:
            print("Terminated: All neighbors have a higher cost (Local Minimum).")
            break

        # Check if we are moving sideways on a plateau
        if lowest_neighbor_cost == current_cost:
            sideways_count += 1
            if sideways_count > max_sideways_moves:
                print("Terminated: Maximum continuous sideways moves reached (Plateau loop).")
                break
        else:
            # Reset counter if we successfully move downward
            sideways_count = 0

        # Choose a random best neighbor to move forward
        next_state = random.choice(neighbors)
        step += 1

        move_type = "Sideways" if lowest_neighbor_cost == current_cost else "Downhill"
        print(f"Step {step}: Moving to {tuple(next_state)} | Cost (h): {lowest_neighbor_cost} ({move_type} Move)")

        current_state = next_state
        current_cost = lowest_neighbor_cost

    if current_cost == 0:
        print("🎉 SUCCESS: Goal State Found!")
        print_board(current_state)
    else:
        print("❌ FAILED: Stuck in a local optimum configuration.")

    return current_state

# --- Execution ---
if __name__ == "__main__":
    # Standard initial configuration from the manual layout: (3, 1, 2, 0)
    initial_configuration = [3, 1, 2, 0]

    # Run the algorithm
    final_state = hill_climbing_4_queens(initial_configuration)


--- Start Initial State ---
State: (3, 1, 2, 0) | Cost (h): 2
 .  .  .  Q 
 .  Q  .  . 
 .  .  Q  . 
 Q  .  .  . 

Step 1: Moving to (3, 0, 2, 0) | Cost (h): 2 (Sideways Move)
Step 2: Moving to (3, 0, 2, 1) | Cost (h): 1 (Downhill Move)
Step 3: Moving to (3, 0, 3, 1) | Cost (h): 1 (Sideways Move)
Step 4: Moving to (2, 0, 3, 1) | Cost (h): 0 (Downhill Move)
🎉 SUCCESS: Goal State Found!
 .  Q  .  . 
 .  .  .  Q 
 Q  .  .  . 
 .  .  Q  . 

